# The Gender Conversion Gap: From STEM and Digital Participation to Labour-Market Power
notebook 00

Table of Content:

- [Project framing](#project-framing)

## feasibility check

 Work + STEM/digital pipeline
 
- OECD wage gap
- part-time gap
- labour force participation gap
- STEM graduates
- coding gap
- women on boards
- missingness check
- 3 possible tests

## <a id="project-framing"></a>Project framing

## Working hypothesises to check

Project topic: The Gender Conversion Gap: From STEM and Digital Participation to Labour-Market Power.

Core research question:
Do stronger STEM and digital participation indicators for women translate into better labour-market and leadership outcomes?

The project will use Estonia as the focal country and compare it against the available OECD-linked cross-country sample.

### Working hypotheses:

### H1 — STEM-to-leadership conversion
Research hypothesis: Countries with higher shares of women among tertiary STEM graduates tend to have higher shares of women in senior and middle management positions.
H0: There is no association between women’s share among tertiary STEM graduates and women’s share in senior/middle management.
Variables: IV = stem_women_share; DV = women_senior_middle_mgmt_share.
Expected direction: positive association.
Test idea: Pearson/Spearman correlation; possibly simple linear regression

#### H2 — STEM-to-pay equality
Research hypothesis: Countries with higher shares of women among tertiary STEM graduates tend to have lower median gender wage gaps.
H0: There is no association between women’s share among tertiary STEM graduates and the median gender wage gap.
Variables: IV = stem_women_share; DV = gender_wage_gap_median.
Expected direction: negative association.
Test idea: Pearson/Spearman correlation; possibly simple linear regression

#### H3 — Digital-skills gap and labour-market participation
Research hypothesis: Countries with larger male advantages in youth coding skills tend to have larger gender gaps in labour-force participation.
H0: There is no association between the youth coding gender gap and the labour-force participation gender gap.
Variables: IV = coding_gap_youth_M_minus_F; DV = labour_force_participation_gap_M_minus_F.
Expected direction: positive association, interpreted cautiously because the age groups differ.
Test idea: Pearson/Spearman correlation; scatterplot; sensitivity check by sample

#### H4 — Work-structure bottleneck
Research hypothesis: Countries where women are more concentrated in part-time employment relative to men tend to have larger median gender wage gaps.
H0: There is no association between the gender gap in part-time employment incidence and the median gender wage gap.
Variables: IV = part_time_employment_gap_F_minus_M; DV = gender_wage_gap_median.
Expected direction: positive association.
Test idea: Pearson/Spearman correlation; possible comparison with STEM-to-wage relationship

H1–H3 are the primary hypotheses. H4 is a secondary structural-bottleneck hypothesis that helps test whether labour-market structure may matter more directly for pay inequality than education or digital pipeline indicators alone.

## Working topic
The gender conversion gap: from STEM/digital participation to labour-market power.

## Goal
Check whether OECD Gender Dashboard indicators provide enough country-level data to test whether education/digital participation indicators are associated with labour-market and leadership outcomes.

## Feasibility criteria
1. Estonia is available.
2. OECD countries have sufficient coverage.
3. Indicators can be joined by country and latest available year.
4. At least 3 testable hypotheses can be formed.
5. Results can support actionable recommendations for Estonia-facing gender-equality stakeholders.
bottleneck:

education / skills → labour market → leadership / pay


## core vars

| Layer                       | Variable                            | Type                  |
| --------------------------- | ----------------------------------- | --------------------- |
| Education pipeline          | women among tertiary STEM graduates | direct                |
| Digital skills pipeline     | coding gap among 16–24              | computed: M - F       |
| Work access                 | labour force participation gap      | computed: M - F       |
| Pay outcome                 | gender wage gap                     | direct                |
| Work structure / care proxy | part-time employment gap            | computed: women - men |
| Leadership proxy            | women in senior/middle management   | proxy                 |


In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from scipy import stats

import requests

from IPython.display import display

from io import StringIO

In [2]:
api_urls = {
    # direct: OBS_VALUE = share of women among tertiary STEM graduates
    "stem_graduates_women_share": "https://sdmx.oecd.org/public/rest/data/OECD.EDU.IMEP,DSD_EAG_UOE_NON_FIN_STUD@DF_UOE_NF_SHARE_FIELD/.ISCED11_5T8.GRAD.FE._T.F05T07._T.A._Z._Z.INST_EDU._T.PT_ST_SUB.F._T?startPeriod=2015&format=csvfile",

    # compute: male OBS_VALUE - female OBS_VALUE
    "coding_gap_youth": "https://sdmx.oecd.org/public/rest/data/OECD.STI.DEP,DSD_ICT_HH_IND@DF_IND/.A.H1K_I.PT_POP._Z.Y16T24.F+M._T._T._T?startPeriod=2015&format=csvfile",

    # compute: male OBS_VALUE - female OBS_VALUE
    # OECD Gender Dashboard defines this indicator for ages 15-74.
    "labour_force_participation_gap": "https://sdmx.oecd.org/public/rest/data/OECD.ELS.SAE,DSD_LFS@DF_LFS_INDIC/.LF_RATE.PT_POP_SUB.M+F.Y15T74.LF?startPeriod=2015&format=csvfile",

    # direct: OBS_VALUE = median gender wage gap
    "gender_wage_gap": "https://sdmx.oecd.org/public/rest/data/OECD.ELS.SAE,DSD_EARNINGS@GENDER_WAGE_GAP/.GWP.PT_WG_SAL_M_D._Z._Z.MEDIAN._T?startPeriod=2015&format=csvfile",

    # compute: part-time incidence by sex = PT / (PT + FT), then women - men
    "part_time_employment_gap": "https://sdmx.oecd.org/public/rest/data/OECD.ELS.SAE,DSD_FTPT@DF_FTPT_COMMON/.EMP.PS.M+F.Y15T64.EMP.MAIN.ICSE93_1.FT+PT.OECD_DEF.A?startPeriod=2015&format=csvfile",

    # proxy, not literal boards: OBS_VALUE = women in senior/middle management
    "women_senior_middle_management_share": "https://sdmx.oecd.org/public/rest/data/OECD.WISE.RSB,DSD_SDG@DF_SDG_G_5/.G_5.5_5.C050502.5_5_2_IC_GEN_MGTN_19ICLS.Y_GE15.F._T._T._T.PT_W?startPeriod=2015&format=csvfile",
}


## Candidate indicators - data dict

This feasibility check uses OECD SDMX API endpoints to evaluate whether the topic can be studied quantitatively.

| Indicator | Role | Construction | Interpretation |
|---|---|---|---|
| stem_graduates_women_share | STEM pipeline | Direct OECD value | Higher = larger share of women among tertiary STEM graduates |
| coding_gap_youth | Digital skills gap | Male value - female value | Higher = larger male advantage in coding skills |
| labour_force_participation_gap | Work access gap | Male value - female value | Higher = larger male advantage in labour force participation |
| gender_wage_gap | Pay outcome | Direct OECD value | Higher = larger median earnings gap |
| part_time_employment_gap | Work-structure gap | Female part-time incidence - male part-time incidence | Higher = women are more concentrated in part-time work |
| women_senior_middle_management_share | Leadership proxy | Direct SDG/OECD value | Higher = larger share of women in senior/middle management |

In [3]:
indicator_meta = pd.DataFrame([
    {
        "layer": "education_pipeline",
        "name": "stem_women_share",
        "source": "stem_graduates_women_share",
        "MEASURE": "GRAD",
        "MEASURE_label": "Graduates",
        "UNIT_MEASURE": "PT_ST_SUB",
        "unit_label": "Percentage of students in the same subgroup",
        "required_filters": "EDUCATION_LEV=ISCED11_5T8; EDUCATION_FIELD=F05T07; SEX=F; MOBILITY=_T",
        "calculation": "Use OBS_VALUE directly",
        "final_unit": "percent",
        "definition": "Women as a percentage of tertiary STEM graduates.",
    },
    {
        "layer": "digital_skills_pipeline",
        "name": "coding_gap_youth_M_minus_F",
        "source": "coding_gap_youth",
        "MEASURE": "H1K_I",
        "MEASURE_label": "Individuals who have written computer code - last 12 m",
        "UNIT_MEASURE": "PT_POP",
        "unit_label": "Percentage of population",
        "required_filters": "AGE=Y16T24; SEX in [M, F]",
        "calculation": "Male OBS_VALUE - female OBS_VALUE",
        "final_unit": "percentage points",
        "definition": "Gender gap in the share of 16-24-year-olds who wrote computer code in the last 12 months.",
    },
    {
        "layer": "work_access",
        "name": "labour_force_participation_gap_M_minus_F",
        "source": "labour_force_participation_gap",
        "MEASURE": "LF_RATE",
        "MEASURE_label": "Labour force participation rate",
        "UNIT_MEASURE": "PT_POP_SUB",
        "unit_label": "Percentage of population in the same subgroup",
        "required_filters": "AGE=Y15T74; LABOUR_FORCE_STATUS=LF; SEX in [M, F]",
        "calculation": "Male OBS_VALUE - female OBS_VALUE",
        "final_unit": "percentage points",
        "definition": "Gender gap in labour force participation rate, ages 15-74.",
    },
    {
        "layer": "pay_outcome",
        "name": "gender_wage_gap_median",
        "source": "gender_wage_gap",
        "MEASURE": "GWP",
        "MEASURE_label": "Gender wage gap",
        "UNIT_MEASURE": "PT_WG_SAL_M_D",
        "unit_label": "Percentage of wages of men in the same decile",
        "required_filters": "AGGREGATION_OPERATION=MEDIAN",
        "calculation": "Use OBS_VALUE directly",
        "final_unit": "percent",
        "definition": "Median gender wage gap, expressed relative to men's wages.",
    },
    {
        "layer": "work_structure_care",
        "name": "part_time_employment_gap_F_minus_M",
        "source": "part_time_employment_gap",
        "MEASURE": "EMP",
        "MEASURE_label": "Employment",
        "UNIT_MEASURE": "PS",
        "unit_label": "Persons",
        "required_filters": "AGE=Y15T64; LABOUR_FORCE_STATUS=EMP; WORK_TIME_ARNGMNT in [FT, PT]; SEX in [M, F]",
        "calculation": "For each sex: PT / (FT + PT) * 100; then female incidence - male incidence",
        "final_unit": "percentage points",
        "definition": "Gender gap in the incidence of part-time employment among employees.",
    },
    {
        "layer": "leadership_proxy",
        "name": "women_senior_middle_mgmt_share",
        "source": "women_senior_middle_management_share",
        "MEASURE": None,
        "SDG_SERIES": "5_5_2_IC_GEN_MGTN_19ICLS",
        "SDG_SERIES_label": "Proportion of women in senior and middle management positions - 19th ICLS",
        "UNIT_MEASURE": "PT_W",
        "unit_label": "Percentage of women",
        "required_filters": "AGE=Y_GE15; SEX=F",
        "calculation": "Use OBS_VALUE directly",
        "final_unit": "percent",
        "definition": "Women as a percentage of senior and middle management positions.",
    },
])

indicator_meta


,layer,name,source,MEASURE,MEASURE_label,UNIT_MEASURE,unit_label,required_filters,calculation,final_unit,definition,SDG_SERIES,SDG_SERIES_label
0,education_pipeline,stem_women_share,stem_graduates_women_share,GRAD,Graduates,PT_ST_SUB,Percentage of students in the same subgroup,EDUCATION_LEV=ISCED11_5T8; EDUCATION_FIELD=F05...,Use OBS_VALUE directly,percent,Women as a percentage of tertiary STEM graduates.,NaN,NaN
1,digital_skills_pipeline,coding_gap_youth_M_minus_F,coding_gap_youth,H1K_I,Individuals who have written computer code - l...,PT_POP,Percentage of population,"AGE=Y16T24; SEX in [M, F]",Male OBS_VALUE - female OBS_VALUE,percentage points,Gender gap in the share of 16-24-year-olds who...,NaN,NaN
2,work_access,labour_force_participation_gap_M_minus_F,labour_force_participation_gap,LF_RATE,Labour force participation rate,PT_POP_SUB,Percentage of population in the same subgroup,"AGE=Y15T74; LABOUR_FORCE_STATUS=LF; SEX in [M, F]",Male OBS_VALUE - female OBS_VALUE,percentage points,"Gender gap in labour force participation rate,...",NaN,NaN
3,pay_outcome,gender_wage_gap_median,gender_wage_gap,GWP,Gender wage gap,PT_WG_SAL_M_D,Percentage of wages of men in the same decile,AGGREGATION_OPERATION=MEDIAN,Use OBS_VALUE directly,percent,"Median gender wage gap, expressed relative to ...",NaN,NaN
4,work_structure_care,part_time_employment_gap_F_minus_M,part_time_employment_gap,EMP,Employment,PS,Persons,AGE=Y15T64; LABOUR_FORCE_STATUS=EMP; WORK_TIME...,For each sex: PT / (FT + PT) * 100; then femal...,percentage points,Gender gap in the incidence of part-time emplo...,NaN,NaN
5,leadership_proxy,women_senior_middle_mgmt_share,women_senior_middle_management_share,None,NaN,PT_W,Percentage of women,AGE=Y_GE15; SEX=F,Use OBS_VALUE directly,percent,Women as a percentage of senior and middle man...,5_5_2_IC_GEN_MGTN_19ICLS,Proportion of women in senior and middle manag...


In [4]:
indicator_enrichment = pd.DataFrame([
    {
        "name": "stem_women_share",
        "indicator_type": "direct_indicator",
        "direction": "Higher = more women represented among tertiary STEM graduates",
        "limitations": (
            "Captures graduation output in selected STEM fields. Does not measure later STEM employment, "
            "retention in technical careers, or quality of career progression."
        ),
    },
    {
        "name": "coding_gap_youth_M_minus_F",
        "indicator_type": "computed_gender_gap",
        "direction": "Higher = larger male advantage in youth coding activity",
        "limitations": (
            "Measures coding activity among 16-24-year-olds, not coding proficiency, formal ICT education, "
            "employment in technology, or later labour-market outcomes."
        ),
    },
    {
        "name": "labour_force_participation_gap_M_minus_F",
        "indicator_type": "computed_gender_gap",
        "direction": "Higher = larger male advantage in labour-force participation",
        "limitations": (
            "Measures labour-force participation for ages 15-74, not employment quality, occupation, earnings, "
            "seniority, job security, or unpaid care responsibilities."
        ),
    },
    {
        "name": "gender_wage_gap_median",
        "indicator_type": "direct_indicator",
        "direction": "Higher = larger wage inequality / worse pay outcome for women",
        "limitations": (
            "Unadjusted wage gap indicator. Should not be interpreted as equal pay for equal work. "
            "May reflect occupational sorting, hours worked, sector, seniority, career interruptions, "
            "and other structural differences."
        ),
    },
    {
        "name": "part_time_employment_gap_F_minus_M",
        "indicator_type": "computed_incidence_gap",
        "direction": "Higher = women are more concentrated in part-time employment relative to men",
        "limitations": (
            "Measures the gender difference in part-time employment incidence. Does not distinguish voluntary "
            "from involuntary part-time work and does not directly measure unpaid care responsibilities, "
            "job quality, precarity, or career penalties."
        ),
    },
    {
        "name": "women_senior_middle_mgmt_share",
        "indicator_type": "direct_indicator_used_as_proxy",
        "direction": "Higher = more women represented in senior and middle management positions",
        "limitations": (
            "Used as a proxy for leadership and labour-market power. Does not measure women on company boards "
            "directly and does not capture sector, firm size, decision-making authority, pay, or seniority "
            "within management categories."
        ),
    },
])

In [5]:
indicator_meta = indicator_meta.merge(
    indicator_enrichment,
    on="name",
    how="left",
    validate="one_to_one"
)

In [6]:
preferred_order = [
    "layer",
    "name",
    "indicator_type",
    "source",
    "MEASURE",
    "MEASURE_label",
    "SDG_SERIES",
    "SDG_SERIES_label",
    "UNIT_MEASURE",
    "unit_label",
    "required_filters",
    "calculation",
    "final_unit",
    "definition",
    "direction",
    "limitations",
]

existing_order = [col for col in preferred_order if col in indicator_meta.columns]
remaining_cols = [col for col in indicator_meta.columns if col not in existing_order]

indicator_meta = indicator_meta[existing_order + remaining_cols]

In [7]:
# OECD member-country flag for EDA sample choices.
# Source: OECD Members and partners page, 38 Member countries.
oecd_member_codes = {
    "AUS", "AUT", "BEL", "CAN", "CHL", "COL", "CRI", "CZE", "DNK", "EST",
    "FIN", "FRA", "DEU", "GRC", "HUN", "ISL", "IRL", "ISR", "ITA", "JPN",
    "KOR", "LVA", "LTU", "LUX", "MEX", "NLD", "NZL", "NOR", "POL", "PRT",
    "SVK", "SVN", "ESP", "SWE", "CHE", "TUR", "GBR", "USA",
}

aggregate_area_codes = {
    "OECD", "OECD_REP", "G7", "EU27", "EU25", "EU19OECD", "EU22OECD",
}

def add_country_scope_flags(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["is_oecd_member"] = out["REF_AREA"].isin(oecd_member_codes)
    out["is_aggregate_area"] = out["REF_AREA"].isin(aggregate_area_codes)
    out["sample_scope"] = np.select(
        [out["is_oecd_member"], out["is_aggregate_area"]],
        ["OECD member", "aggregate area"],
        default="non-member / partner / candidate",
    )
    return out



In [8]:
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

def fetch_oecd_csv(url: str) -> pd.DataFrame:
    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()
    return pd.read_csv(StringIO(resp.text))

raw_data = {}

for name, url in api_urls.items():
    print(f"\nLoading {name}")
    df = fetch_oecd_csv(url)
    raw_data[name] = df
    print("  shape:", df.shape)


Loading stem_graduates_women_share
  shape: (443, 28)

Loading coding_gap_youth


  shape: (512, 20)

Loading labour_force_participation_gap
  shape: (1240, 12)

Loading gender_wage_gap
  shape: (343, 14)

Loading part_time_employment_gap
  shape: (2204, 17)

Loading women_senior_middle_management_share


  shape: (261, 19)


In [9]:
for name, df in raw_data.items():
    print(f"\n=== {name} ===")
    display(df.head())
    print("\nDtypes:")
    display(df.dtypes)
    print("\nTop 10 columns by missingness:")
    display(df.isna().mean().sort_values(ascending=False).head(10))


=== stem_graduates_women_share ===


,DATAFLOW,REF_AREA,EDUCATION_LEV,MEASURE,EDUCATION_TYPE,INTENSITY,EDUCATION_FIELD,GRADE,FREQ,ORIGIN,...,REF_YEAR_AGES,ORIGIN_CRITERION,REPYEARSTART,REPYEAREND,OBS_STATUS,CONF_STATUS,COMMENT_OBS,DECIMALS,TIME_PER_COLLECT,UNIT_MULT
0,OECD.EDU.IMEP:DSD_EAG_UOE_NON_FIN_STUD@DF_UOE_...,PER,ISCED11_5T8,GRAD,FE,_T,F05T07,_T,A,_Z,...,NaN,NaN,NaN,NaN,O,NaN,NaN,2,NaN,0
1,OECD.EDU.IMEP:DSD_EAG_UOE_NON_FIN_STUD@DF_UOE_...,PER,ISCED11_5T8,GRAD,FE,_T,F05T07,_T,A,_Z,...,NaN,NaN,NaN,NaN,O,NaN,NaN,2,NaN,0
2,OECD.EDU.IMEP:DSD_EAG_UOE_NON_FIN_STUD@DF_UOE_...,PER,ISCED11_5T8,GRAD,FE,_T,F05T07,_T,A,_Z,...,NaN,NaN,NaN,NaN,O,NaN,NaN,2,NaN,0
3,OECD.EDU.IMEP:DSD_EAG_UOE_NON_FIN_STUD@DF_UOE_...,ISR,ISCED11_5T8,GRAD,FE,_T,F05T07,_T,A,_Z,...,NaN,NaN,NaN,NaN,O,NaN,NaN,2,NaN,0
4,OECD.EDU.IMEP:DSD_EAG_UOE_NON_FIN_STUD@DF_UOE_...,ISR,ISCED11_5T8,GRAD,FE,_T,F05T07,_T,A,_Z,...,NaN,NaN,NaN,NaN,O,NaN,NaN,2,NaN,0



Dtypes:


DATAFLOW             object
REF_AREA             object
EDUCATION_LEV        object
MEASURE              object
EDUCATION_TYPE       object
INTENSITY            object
EDUCATION_FIELD      object
GRADE                object
FREQ                 object
ORIGIN               object
DESTINATION          object
INST_TYPE_EDU        object
MOBILITY             object
UNIT_MEASURE         object
SEX                  object
AGE                  object
TIME_PERIOD           int64
OBS_VALUE           float64
REF_YEAR_AGES       float64
ORIGIN_CRITERION    float64
REPYEARSTART        float64
REPYEAREND          float64
OBS_STATUS           object
CONF_STATUS         float64
COMMENT_OBS         float64
DECIMALS              int64
TIME_PER_COLLECT    float64
UNIT_MULT             int64
dtype: object


Top 10 columns by missingness:


...

,DATAFLOW,REF_AREA,SDG_GOAL,SDG_TARGET,SDG_INDICATOR,SDG_SERIES,AGE,SEX,INCOME_WEALTH_QUANTILE,EDUCATION_LEV,DEG_URB,UNIT_MEASURE,TIME_PERIOD,OBS_VALUE,SOURCE,UNIT_MULT,PRICE_BASE,BASE_PER,DECIMALS
0,OECD.WISE.RSB:DSD_SDG@DF_SDG_G_5(2.0),AUT,G_5,5_5,C050502,5_5_2_IC_GEN_MGTN_19ICLS,Y_GE15,F,_T,_T,_T,PT_W,2023,33.79,UNSD,0,NaN,NaN,2
1,OECD.WISE.RSB:DSD_SDG@DF_SDG_G_5(2.0),AUS,G_5,5_5,C050502,5_5_2_IC_GEN_MGTN_19ICLS,Y_GE15,F,_T,_T,_T,PT_W,2024,39.39,UNSD,0,NaN,NaN,2
2,OECD.WISE.RSB:DSD_SDG@DF_SDG_G_5(2.0),AUS,G_5,5_5,C050502,5_5_2_IC_GEN_MGTN_19ICLS,Y_GE15,F,_T,_T,_T,PT_W,2023,38.90,UNSD,0,NaN,NaN,2
3,OECD.WISE.RSB:DSD_SDG@DF_SDG_G_5(2.0),SVN,G_5,5_5,C050502,5_5_2_IC_GEN_MGTN_19ICLS,Y_GE15,F,_T,_T,_T,PT_W,2022,33.17,UNSD,0,NaN,NaN,2
4,OECD.WISE.RSB:DSD_SDG@DF_SDG_G_5(2.0),ESP,G_5,5_5,C050502,5_5_2_IC_GEN_MGTN_19ICLS,Y_GE15,F,_T,_T,_T,PT_W,2020,37.49,UNSD,0,NaN,NaN,2



Dtypes:


DATAFLOW                   object
REF_AREA                   object
SDG_GOAL                   object
SDG_TARGET                 object
SDG_INDICATOR              object
SDG_SERIES                 object
AGE                        object
SEX                        object
INCOME_WEALTH_QUANTILE     object
EDUCATION_LEV              object
DEG_URB                    object
UNIT_MEASURE               object
TIME_PERIOD                 int64
OBS_VALUE                 float64
SOURCE                     object
UNIT_MULT                   int64
PRICE_BASE                float64
BASE_PER                  float64
DECIMALS                    int64
dtype: object


Top 10 columns by missingness:


BASE_PER         1.0
PRICE_BASE       1.0
DATAFLOW         0.0
DEG_URB          0.0
UNIT_MULT        0.0
SOURCE           0.0
OBS_VALUE        0.0
TIME_PERIOD      0.0
UNIT_MEASURE     0.0
EDUCATION_LEV    0.0
dtype: float64

## func of prerparation data

In [10]:
def check_country_year_duplicates(df, value_name="OBS_VALUE"):
    dupes = (
        df.groupby(["REF_AREA", "TIME_PERIOD"])
          .size()
          .reset_index(name="n")
          .query("n > 1")
          .sort_values("n", ascending=False)
    )
    
    print(f"Duplicate country-year rows: {len(dupes)}")
    display(dupes.head(10))
    return dupes

In [11]:
def apply_exact_filters(df: pd.DataFrame, filters: dict | None = None) -> pd.DataFrame:
    """Apply simple equality or isin filters used to pin an OECD SDMX slice."""
    out = df.copy()

    if filters:
        for col, val in filters.items():
            if col not in out.columns:
                raise KeyError(f"Filter column not found: {col}")
            if isinstance(val, (list, tuple, set)):
                out = out[out[col].isin(val)]
            else:
                out = out[out[col] == val]

    return out


def country_year_duplicates(df: pd.DataFrame, keys: list[str]) -> pd.DataFrame:
    return (
        df.groupby(keys)
          .size()
          .reset_index(name="n")
          .query("n > 1")
          .sort_values("n", ascending=False)
    )


def prepare_direct_indicator(
    df: pd.DataFrame,
    value_name: str,
    extra_filters: dict | None = None,
) -> pd.DataFrame:
    df_clean = apply_exact_filters(df, extra_filters)
    df_clean = df_clean.dropna(subset=["OBS_VALUE"])

    dupes = country_year_duplicates(df_clean, ["REF_AREA", "TIME_PERIOD"])
    if len(dupes) > 0:
        print(f"Warning: {value_name} has duplicate country-year rows after filtering.")
        display(dupes.head())

    out = df_clean[["REF_AREA", "TIME_PERIOD", "OBS_VALUE"]].rename(
        columns={"OBS_VALUE": value_name}
    )

    return out



In [12]:
def prepare_gap_indicator(
    df: pd.DataFrame,
    gap_name: str,
    direction: str = "M-F",
    extra_filters: dict | None = None,
) -> pd.DataFrame:
    """
    Build a sex gap from OECD rows.
    - direction="M-F" -> male - female
    - direction="F-M" -> female - male
    Output is in the same unit as OBS_VALUE, usually percentage points.
    """
    tmp = apply_exact_filters(df, extra_filters)
    tmp = tmp.dropna(subset=["OBS_VALUE"])

    dupes = country_year_duplicates(tmp, ["REF_AREA", "TIME_PERIOD", "SEX"])
    if len(dupes) > 0:
        print(f"Warning: {gap_name} has duplicate country-year-sex rows after filtering.")
        display(dupes.head())

    pivot = (
        tmp.pivot_table(
            index=["REF_AREA", "TIME_PERIOD"],
            columns="SEX",
            values="OBS_VALUE",
            aggfunc="mean",
        )
        .reset_index()
    )

    available_sexes = [c for c in pivot.columns if c not in ["REF_AREA", "TIME_PERIOD"]]
    if "F" not in available_sexes or "M" not in available_sexes:
        print(f"Warning in {gap_name}: cannot find both F and M in columns {available_sexes}")
        return pivot

    if direction == "M-F":
        pivot[gap_name] = pivot["M"] - pivot["F"]
    elif direction == "F-M":
        pivot[gap_name] = pivot["F"] - pivot["M"]
    else:
        raise ValueError("direction must be 'M-F' or 'F-M'")

    return pivot[["REF_AREA", "TIME_PERIOD", gap_name]].dropna(subset=[gap_name])



In [13]:
def prepare_part_time_gap(
    df: pd.DataFrame,
    extra_filters: dict | None = None,
) -> pd.DataFrame:
    """
    Compute part-time employment incidence gap (F - M), in percentage points.

    Uses FTPT dataset:
    - WORK_TIME_ARNGMNT in {'PT', 'FT'}
    - MEASURE == 'EMP', UNIT_MEASURE == 'PS'
    - AGE == 'Y15T64', LABOUR_FORCE_STATUS == 'EMP'
    - JOB_COVERAGE == 'MAIN', WORKER_STATUS == 'ICSE93_1'
    - METHODOLOGY == 'OECD_DEF'
    """
    base_filters = {
        "MEASURE": "EMP",
        "UNIT_MEASURE": "PS",
        "AGE": "Y15T64",
        "LABOUR_FORCE_STATUS": "EMP",
        "JOB_COVERAGE": "MAIN",
        "WORKER_STATUS": "ICSE93_1",
        "METHODOLOGY": "OECD_DEF",
    }
    tmp = apply_exact_filters(df, base_filters)
    tmp = apply_exact_filters(tmp, extra_filters)

    needed_cols = ["REF_AREA", "TIME_PERIOD", "SEX", "WORK_TIME_ARNGMNT", "OBS_VALUE"]
    tmp = tmp[needed_cols].dropna(subset=["OBS_VALUE"])

    dupes = country_year_duplicates(tmp, ["REF_AREA", "TIME_PERIOD", "SEX", "WORK_TIME_ARNGMNT"])
    if len(dupes) > 0:
        print("Warning: part-time data has duplicate country-year-sex-worktime rows after filtering.")
        display(dupes.head())

    pivot = (
        tmp.pivot_table(
            index=["REF_AREA", "TIME_PERIOD", "SEX"],
            columns="WORK_TIME_ARNGMNT",
            values="OBS_VALUE",
            aggfunc="mean",
        )
        .reset_index()
    )

    if "FT" not in pivot.columns or "PT" not in pivot.columns:
        print("Warning: cannot find both FT and PT columns.")
        print(pivot.columns.tolist())
        return pivot

    pivot["part_time_incidence_pct"] = pivot["PT"] / (pivot["FT"] + pivot["PT"]) * 100

    sex_pivot = (
        pivot.pivot_table(
            index=["REF_AREA", "TIME_PERIOD"],
            columns="SEX",
            values="part_time_incidence_pct",
            aggfunc="mean",
        )
        .reset_index()
    )

    if "F" not in sex_pivot.columns or "M" not in sex_pivot.columns:
        print("Warning: cannot find both F and M after sex pivot.")
        return sex_pivot

    sex_pivot["part_time_employment_gap_F_minus_M"] = sex_pivot["F"] - sex_pivot["M"]

    return sex_pivot[
        ["REF_AREA", "TIME_PERIOD", "part_time_employment_gap_F_minus_M"]
    ].dropna()



In [14]:
# part‑time employment gap: women − men

pt_gap = prepare_part_time_gap(
    raw_data["part_time_employment_gap"],
    extra_filters={
        "AGE": "Y15T64",
        "LABOUR_FORCE_STATUS": "EMP",
        # add more filters after inspecting real column values
    }
)

print(pt_gap.head())
pt_gap.shape

SEX REF_AREA  TIME_PERIOD  part_time_employment_gap_F_minus_M
0        ARG         2016                           27.585443
1        ARG         2017                           24.653910
2        ARG         2018                           27.335464
3        ARG         2019                           26.545918
4        AUS         2015                           23.643429


(551, 3)

In [15]:
# STEM: share of women among tertiary STEM graduates
stem = prepare_direct_indicator(
    raw_data["stem_graduates_women_share"],
    value_name="stem_women_share",
    extra_filters={
        # add more filters if needed
        # 'AGE': '_T',
        # 'EDUCATION_FIELD': 'F05T07',
    },
)
stem.head()

,REF_AREA,TIME_PERIOD,stem_women_share
22,EU25,2015,33.036127
23,EU25,2016,33.439321
24,EU25,2017,33.366606
25,EU25,2018,33.466066
26,EU25,2019,33.288263


In [16]:
# Coding gap 16–24: M − F
coding_gap = prepare_gap_indicator(
    raw_data["coding_gap_youth"],
    gap_name="coding_gap_youth_M_minus_F",
    direction="M-F",
    extra_filters={
        "AGE": "Y16T24",
        # add more filters if needed:
        # "EDUCATION_LEVEL": "_T",
        # "INCOME_GROUP": "_T",
        # "EMP_STATUS": "_T",
    },
)
coding_gap.head()

SEX,REF_AREA,TIME_PERIOD,coding_gap_youth_M_minus_F
0,AUT,2015,19.1709
1,AUT,2016,8.8533
2,AUT,2017,16.2118
3,AUT,2019,16.9658
4,AUT,2021,13.0546


In [17]:
# Labour force participation gap: M − F

lfp_gap = prepare_gap_indicator(
    raw_data["labour_force_participation_gap"],
    gap_name="labour_force_participation_gap_M_minus_F",
    direction="M-F",
    extra_filters={
        "AGE": "Y15T74",
        "LABOUR_FORCE_STATUS": "LF",
    },
)
lfp_gap.head()


SEX,REF_AREA,TIME_PERIOD,labour_force_participation_gap_M_minus_F
0,ARG,2016,23.104
1,ARG,2017,22.902
2,ARG,2018,21.494
3,ARG,2019,20.409
4,ARG,2020,21.215


In [18]:
# Gender wage gap: direct

wage_gap = prepare_direct_indicator(
    raw_data["gender_wage_gap"],
    value_name="gender_wage_gap_median",
    extra_filters={
        "SEX": "_T",
        "AGGREGATION_OPERATION": "MEDIAN",
    },
)
wage_gap.head()

,REF_AREA,TIME_PERIOD,gender_wage_gap_median
0,AUS,2015,13.307692
1,AUS,2016,13.728432
2,AUS,2017,13.000948
3,AUS,2018,13.448567
4,AUS,2019,15.311653


In [19]:
# Women in senior/middle management: direct

management = prepare_direct_indicator(
    raw_data["women_senior_middle_management_share"],
    value_name="women_senior_middle_mgmt_share",
    extra_filters={
        "SEX": "F",
        "AGE": "Y_GE15",
        # SDG_SERIES задаёт именно management‑proxy показатель:
        "SDG_SERIES": "5_5_2_IC_GEN_MGTN_19ICLS",
    },
)
management.head()

,REF_AREA,TIME_PERIOD,women_senior_middle_mgmt_share
0,AUT,2023,33.79
1,AUS,2024,39.39
2,AUS,2023,38.90
3,SVN,2022,33.17
4,ESP,2020,37.49


In [20]:
for name, d in [
    ("stem", stem),
    ("coding_gap", coding_gap),
    ("lfp_gap", lfp_gap),
    ("wage_gap", wage_gap),
    ("pt_gap", pt_gap),
    ("management", management),
]:
    print(
        f"{name:12} shape={d.shape}, has EST={d['REF_AREA'].eq('EST').any()}"
    )

stem         shape=(421, 3), has EST=True
coding_gap   shape=(256, 3), has EST=True
lfp_gap      shape=(620, 3), has EST=True
wage_gap     shape=(343, 3), has EST=True
pt_gap       shape=(551, 3), has EST=True
management   shape=(261, 3), has EST=True


## Mini check for all indicators


In [21]:
prepared = {
    "stem_graduates_women_share": stem,
    "coding_gap_youth_M_minus_F": coding_gap,
    "labour_force_participation_gap_M_minus_F": lfp_gap,
    "gender_wage_gap": wage_gap,
    "part_time_employment_gap_F_minus_M": pt_gap,
    "women_senior_middle_management_share": management,
}

In [22]:
raw_coverage_rows = []

for source_name, df in raw_data.items():
    raw_coverage_rows.append({
        "source": source_name,
        "raw_rows": len(df),
        "raw_countries": df["REF_AREA"].nunique(),
        "raw_min_year": df["TIME_PERIOD"].min(),
        "raw_max_year": df["TIME_PERIOD"].max(),
        "raw_has_estonia": "EST" in set(df["REF_AREA"]),
        "raw_obs_missing_rate": df["OBS_VALUE"].isna().mean(),
    })

raw_coverage = pd.DataFrame(raw_coverage_rows)
display(raw_coverage)
print('='*60)

prepared_coverage_rows = []

for name, df in prepared.items():
    value_col = [c for c in df.columns if c not in ["REF_AREA", "TIME_PERIOD"]][0]
    prepared_coverage_rows.append({
        "indicator": name,
        "rows": len(df),
        "countries_all_available": df["REF_AREA"].nunique(),
        "countries_oecd_members": df[df["REF_AREA"].isin(oecd_member_codes)]["REF_AREA"].nunique(),
        "min_year": df["TIME_PERIOD"].min(),
        "max_year": df["TIME_PERIOD"].max(),
        "has_estonia": "EST" in set(df["REF_AREA"]),
        "prepared_missing_rate": df[value_col].isna().mean(),
    })

coverage = pd.DataFrame(prepared_coverage_rows)
display(coverage)

,source,raw_rows,raw_countries,raw_min_year,raw_max_year,raw_has_estonia,raw_obs_missing_rate
0,stem_graduates_women_share,443,46,2015,2024,True,0.049661
1,coding_gap_youth,512,39,2015,2025,True,0.000000
2,labour_force_participation_gap,1240,58,2015,2025,True,0.000000
3,gender_wage_gap,343,50,2015,2024,True,0.000000
4,part_time_employment_gap,2204,52,2015,2025,True,0.000000
5,women_senior_middle_management_share,261,30,2015,2024,True,0.000000


,indicator,rows,countries_all_available,countries_oecd_members,min_year,max_year,has_estonia,prepared_missing_rate
0,stem_graduates_women_share,421,45,38,2015,2024,True,0.0
1,coding_gap_youth_M_minus_F,256,39,32,2015,2025,True,0.0
2,labour_force_participation_gap_M_minus_F,620,58,38,2015,2025,True,0.0
3,gender_wage_gap,343,50,38,2015,2024,True,0.0
4,part_time_employment_gap_F_minus_M,551,52,36,2015,2025,True,0.0
5,women_senior_middle_management_share,261,30,30,2015,2024,True,0.0


In [23]:
# latest value per country

def latest_by_country(df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    tmp = df.copy()
    tmp["TIME_PERIOD"] = pd.to_numeric(tmp["TIME_PERIOD"], errors="coerce")
    
    latest = (
        tmp.dropna(subset=[value_col, "TIME_PERIOD"])
           .sort_values(["REF_AREA", "TIME_PERIOD"])
           .groupby("REF_AREA", as_index=False)
           .tail(1)
           .rename(columns={
               "TIME_PERIOD": f"{value_col}_year"
           })
    )

    return latest[["REF_AREA", f"{value_col}_year", value_col]]

In [24]:
# Merge latest available country-level values into one wide table.

from functools import reduce

latest_tables = []

for name, df in prepared.items():
    value_col = [c for c in df.columns if c not in ["REF_AREA", "TIME_PERIOD"]][0]
    latest_tables.append(latest_by_country(df, value_col))

latest_wide = reduce(
    lambda left, right: pd.merge(left, right, on="REF_AREA", how="outer"),
    latest_tables,
)

latest_wide = add_country_scope_flags(latest_wide)
latest_wide.head()



,REF_AREA,stem_women_share_year,stem_women_share,coding_gap_youth_M_minus_F_year,coding_gap_youth_M_minus_F,labour_force_participation_gap_M_minus_F_year,labour_force_participation_gap_M_minus_F,gender_wage_gap_median_year,gender_wage_gap_median,part_time_employment_gap_F_minus_M_year,part_time_employment_gap_F_minus_M,women_senior_middle_mgmt_share_year,women_senior_middle_mgmt_share,is_oecd_member,is_aggregate_area,sample_scope
0,ARG,NaN,NaN,NaN,NaN,2023.0,19.078,2024.0,9.090909,2019.0,26.545918,NaN,NaN,False,False,non-member / partner / candidate
1,AUS,2024.0,33.352506,NaN,NaN,2025.0,7.650,2024.0,10.671213,2025.0,16.924777,2024.0,39.39,True,False,OECD member
2,AUT,2024.0,29.190229,2025.0,10.8682,2025.0,8.144,2024.0,11.062252,2025.0,25.858781,2023.0,33.79,True,False,OECD member
3,BEL,2024.0,28.368823,2025.0,9.9548,2025.0,7.970,2022.0,0.906909,2025.0,18.302429,2023.0,33.69,True,False,OECD member
4,BGR,2023.0,36.155884,2025.0,2.3700,2025.0,9.686,2024.0,2.360074,2025.0,0.595541,NaN,NaN,False,False,non-member / partner / candidate


In [32]:
latest_wide.columns

Index(['REF_AREA', 'stem_women_share_year', 'stem_women_share',
       'coding_gap_youth_M_minus_F_year', 'coding_gap_youth_M_minus_F',
       'labour_force_participation_gap_M_minus_F_year',
       'labour_force_participation_gap_M_minus_F',
       'gender_wage_gap_median_year', 'gender_wage_gap_median',
       'part_time_employment_gap_F_minus_M_year',
       'part_time_employment_gap_F_minus_M',
       'women_senior_middle_mgmt_share_year', 'women_senior_middle_mgmt_share',
       'is_oecd_member', 'is_aggregate_area', 'sample_scope',
       'latest_year_min', 'latest_year_max', 'latest_year_span'],
      dtype='object')

In [25]:
latest_wide.shape

(60, 16)

In [26]:
# Estonia check

latest_wide[latest_wide["REF_AREA"] == "EST"]

,REF_AREA,stem_women_share_year,stem_women_share,coding_gap_youth_M_minus_F_year,coding_gap_youth_M_minus_F,labour_force_participation_gap_M_minus_F_year,labour_force_participation_gap_M_minus_F,gender_wage_gap_median_year,gender_wage_gap_median,part_time_employment_gap_F_minus_M_year,part_time_employment_gap_F_minus_M,women_senior_middle_mgmt_share_year,women_senior_middle_mgmt_share,is_oecd_member,is_aggregate_area,sample_scope
16,EST,2024.0,41.545704,2025.0,7.5205,2025.0,4.924,2024.0,19.770206,2025.0,5.54288,2023.0,31.51,True,False,OECD member


In [27]:
year_cols = [c for c in latest_wide.columns if c.endswith("_year")]

latest_wide["latest_year_min"] = latest_wide[year_cols].min(axis=1)
latest_wide["latest_year_max"] = latest_wide[year_cols].max(axis=1)
latest_wide["latest_year_span"] = latest_wide["latest_year_max"] - latest_wide["latest_year_min"]

latest_wide[["REF_AREA", "sample_scope", "latest_year_min", "latest_year_max", "latest_year_span"]].sort_values(
    "latest_year_span", ascending=False
).head(10)



,REF_AREA,sample_scope,latest_year_min,latest_year_max,latest_year_span
8,CHL,OECD member,2017.0,2025.0,8.0
48,PER,non-member / partner / candidate,2017.0,2024.0,7.0
25,GBR,OECD member,2019.0,2025.0,6.0
0,ARG,non-member / partner / candidate,2019.0,2024.0,5.0
32,ISL,OECD member,2021.0,2025.0,4.0
33,ISR,OECD member,2022.0,2025.0,3.0
6,CAN,OECD member,2022.0,2025.0,3.0
3,BEL,OECD member,2022.0,2025.0,3.0
35,JPN,OECD member,2022.0,2025.0,3.0
50,PRT,OECD member,2022.0,2025.0,3.0


In [28]:
candidate_pairs = [
    ("stem_women_share", "women_senior_middle_mgmt_share"),
    ("coding_gap_youth_M_minus_F", "gender_wage_gap_median"),
    ("coding_gap_youth_M_minus_F", "labour_force_participation_gap_M_minus_F"),
    ("part_time_employment_gap_F_minus_M", "gender_wage_gap_median"),
    ("stem_women_share", "gender_wage_gap_median"),
    ("stem_women_share", "part_time_employment_gap_F_minus_M"),
]

pairwise_rows = []

for x, y in candidate_pairs:
    x_year = f"{x}_year"
    y_year = f"{y}_year"
    clean = latest_wide[["REF_AREA", "is_oecd_member", x, y, x_year, y_year]].dropna(subset=[x, y])
    clean_oecd = clean[clean["is_oecd_member"]]

    pairwise_rows.append({
        "x": x,
        "y": y,
        "n_all_available": len(clean),
        "n_oecd_members": len(clean_oecd),
        "has_estonia": "EST" in set(clean["REF_AREA"]),
        "max_year_gap_all": (clean[x_year] - clean[y_year]).abs().max(),
        "max_year_gap_oecd": (clean_oecd[x_year] - clean_oecd[y_year]).abs().max(),
    })

pairwise_overlap = pd.DataFrame(pairwise_rows)
pairwise_overlap



,x,y,n_all_available,n_oecd_members,has_estonia,max_year_gap_all,max_year_gap_oecd
0,stem_women_share,women_senior_middle_mgmt_share,30,30,True,1.0,1.0
1,coding_gap_youth_M_minus_F,gender_wage_gap_median,38,32,True,6.0,6.0
2,coding_gap_youth_M_minus_F,labour_force_participation_gap_M_minus_F,38,32,True,8.0,8.0
3,part_time_employment_gap_F_minus_M,gender_wage_gap_median,46,36,True,5.0,3.0
4,stem_women_share,gender_wage_gap_median,44,38,True,5.0,2.0
5,stem_women_share,part_time_employment_gap_F_minus_M,41,36,True,3.0,2.0


In [29]:
# Export analysis-ready tables for downstream EDA and hypothesis testing

from pathlib import Path

processed_dir = Path("data/processed")
raw_dir = Path("data/raw")
processed_dir.mkdir(parents=True, exist_ok=True)
raw_dir.mkdir(parents=True, exist_ok=True)

value_cols = [
    "stem_women_share",
    "coding_gap_youth_M_minus_F",
    "labour_force_participation_gap_M_minus_F",
    "gender_wage_gap_median",
    "part_time_employment_gap_F_minus_M",
    "women_senior_middle_mgmt_share",
]

year_cols = [f"{col}_year" for col in value_cols]
latest_export_cols = [
    "REF_AREA",
    "is_oecd_member",
    "is_aggregate_area",
    "sample_scope",
    *value_cols,
    *year_cols,
    "latest_year_min",
    "latest_year_max",
    "latest_year_span",
]

gender_conversion_gap_latest = latest_wide[latest_export_cols].copy()
gender_conversion_gap_latest.to_csv(processed_dir / "gender_conversion_gap_latest.csv", index=False)

coverage_source_map = {
    "stem_graduates_women_share": "stem_graduates_women_share",
    "coding_gap_youth_M_minus_F": "coding_gap_youth",
    "labour_force_participation_gap_M_minus_F": "labour_force_participation_gap",
    "gender_wage_gap": "gender_wage_gap",
    "part_time_employment_gap_F_minus_M": "part_time_employment_gap",
    "women_senior_middle_management_share": "women_senior_middle_management_share",
}

raw_missingness_summary = raw_coverage.rename(columns={
    "source": "source_key",
    "raw_countries": "n_ref_area_raw",
    "raw_min_year": "min_year_raw",
    "raw_max_year": "max_year_raw",
    "raw_obs_missing_rate": "raw_missing_obs_value_share",
}).copy()
raw_missingness_summary = raw_missingness_summary[[
    "source_key",
    "raw_rows",
    "raw_missing_obs_value_share",
    "n_ref_area_raw",
    "min_year_raw",
    "max_year_raw",
]]
raw_missingness_summary.to_csv(processed_dir / "raw_missingness_summary.csv", index=False)

indicator_coverage = coverage.rename(columns={
    "rows": "prepared_rows",
    "prepared_missing_rate": "prepared_missing_share",
}).copy()
indicator_coverage["source_key"] = indicator_coverage["indicator"].map(coverage_source_map)
indicator_coverage = indicator_coverage.merge(
    raw_missingness_summary[["source_key", "raw_missing_obs_value_share"]],
    on="source_key",
    how="left",
)
indicator_coverage = indicator_coverage[[
    "indicator",
    "source_key",
    "prepared_rows",
    "countries_all_available",
    "countries_oecd_members",
    "min_year",
    "max_year",
    "has_estonia",
    "raw_missing_obs_value_share",
    "prepared_missing_share",
]]
indicator_coverage.to_csv(processed_dir / "indicator_coverage.csv", index=False)

hypothesis_mapping = pd.DataFrame([
    {
        "hypothesis_id": "H1",
        "hypothesis_name": "STEM-to-leadership conversion",
        "research_hypothesis": "Countries with higher shares of women among tertiary STEM graduates tend to have higher shares of women in senior and middle management positions.",
        "null_hypothesis": "There is no association between women’s share among tertiary STEM graduates and women’s share in senior/middle management.",
        "alternative_hypothesis": "There is an association between women’s share among tertiary STEM graduates and women’s share in senior/middle management.",
        "independent_variable": "stem_women_share",
        "dependent_variable": "women_senior_middle_mgmt_share",
        "expected_direction": "positive",
        "sample_note": "Compare all available observations with OECD-member-only sample; aggregate areas should be excluded from country-level tests.",
        "interpretation_note": "Leadership is measured with a senior/middle management proxy, not board representation.",
    },
    {
        "hypothesis_id": "H2",
        "hypothesis_name": "STEM-to-pay equality",
        "research_hypothesis": "Countries with higher shares of women among tertiary STEM graduates tend to have lower median gender wage gaps.",
        "null_hypothesis": "There is no association between women’s share among tertiary STEM graduates and the median gender wage gap.",
        "alternative_hypothesis": "There is an association between women’s share among tertiary STEM graduates and the median gender wage gap.",
        "independent_variable": "stem_women_share",
        "dependent_variable": "gender_wage_gap_median",
        "expected_direction": "negative",
        "sample_note": "Compare all available observations with OECD-member-only sample; aggregate areas should be excluded from country-level tests.",
        "interpretation_note": "The wage gap is unadjusted, so results are descriptive rather than causal.",
    },
    {
        "hypothesis_id": "H3",
        "hypothesis_name": "Digital-skills gap and labour-market participation",
        "research_hypothesis": "Countries with larger male advantages in youth coding skills tend to have larger gender gaps in labour-force participation.",
        "null_hypothesis": "There is no association between the youth coding gender gap and the labour-force participation gender gap.",
        "alternative_hypothesis": "There is an association between the youth coding gender gap and the labour-force participation gender gap.",
        "independent_variable": "coding_gap_youth_M_minus_F",
        "dependent_variable": "labour_force_participation_gap_M_minus_F",
        "expected_direction": "positive",
        "sample_note": "Compare all available observations with OECD-member-only sample; aggregate areas should be excluded from country-level tests.",
        "interpretation_note": "Interpret cautiously because coding is measured for ages 16-24 while labour-force participation is measured for ages 15-74.",
    },
    {
        "hypothesis_id": "H4",
        "hypothesis_name": "Work-structure bottleneck",
        "research_hypothesis": "Countries where women are more concentrated in part-time employment relative to men tend to have larger median gender wage gaps.",
        "null_hypothesis": "There is no association between the gender gap in part-time employment incidence and the median gender wage gap.",
        "alternative_hypothesis": "There is an association between the gender gap in part-time employment incidence and the median gender wage gap.",
        "independent_variable": "part_time_employment_gap_F_minus_M",
        "dependent_variable": "gender_wage_gap_median",
        "expected_direction": "positive",
        "sample_note": "Compare all available observations with OECD-member-only sample; aggregate areas should be excluded from country-level tests.",
        "interpretation_note": "Part-time gap is a work-structure proxy and does not distinguish voluntary from involuntary part-time work.",
    },
])
hypothesis_mapping.to_csv(processed_dir / "hypothesis_mapping.csv", index=False)

pair_to_hypothesis = {
    (row["independent_variable"], row["dependent_variable"]): row["hypothesis_id"]
    for _, row in hypothesis_mapping.iterrows()
}
pairwise_overlap_export = pairwise_overlap.copy()
pairwise_overlap_export["hypothesis"] = pairwise_overlap_export.apply(
    lambda row: pair_to_hypothesis.get((row["x"], row["y"]), "exploratory"),
    axis=1,
)
pairwise_overlap_export = pairwise_overlap_export[[
    "hypothesis",
    "x",
    "y",
    "n_all_available",
    "n_oecd_members",
    "has_estonia",
    "max_year_gap_all",
    "max_year_gap_oecd",
]]
pairwise_overlap_export.to_csv(processed_dir / "pairwise_overlap.csv", index=False)

data_dictionary_cols = [
    "layer",
    "name",
    "source",
    "indicator_type",
    "MEASURE",
    "MEASURE_label",
    "SDG_SERIES",
    "SDG_SERIES_label",
    "UNIT_MEASURE",
    "unit_label",
    "required_filters",
    "calculation",
    "final_unit",
    "definition",
    "direction",
    "limitations",
]
data_dictionary = indicator_meta.copy()
for col in data_dictionary_cols:
    if col not in data_dictionary.columns:
        data_dictionary[col] = np.nan

data_dictionary = data_dictionary[data_dictionary_cols]
data_dictionary.to_csv(processed_dir / "data_dictionary.csv", index=False)

estonia_latest_values = pd.DataFrame([
    {
        "variable": col,
        "latest_year": latest_wide.loc[latest_wide["REF_AREA"] == "EST", f"{col}_year"].iloc[0],
        "estonia_value": latest_wide.loc[latest_wide["REF_AREA"] == "EST", col].iloc[0],
        "unit": data_dictionary.loc[data_dictionary["name"] == col, "final_unit"].iloc[0],
        "direction": data_dictionary.loc[data_dictionary["name"] == col, "direction"].iloc[0],
        "interpretation_note": data_dictionary.loc[data_dictionary["name"] == col, "definition"].iloc[0],
    }
    for col in value_cols
])
estonia_latest_values.to_csv(processed_dir / "estonia_latest_values.csv", index=False)

source_url_registry = pd.DataFrame([
    {
        "source_key": key,
        "api_url": url,
        "role_in_project": {
            "stem_graduates_women_share": "STEM pipeline indicator",
            "coding_gap_youth": "Digital skills pipeline indicator; raw sex-specific values used to compute M-F gap",
            "labour_force_participation_gap": "Work access indicator; raw sex-specific values used to compute M-F gap",
            "gender_wage_gap": "Pay outcome indicator",
            "part_time_employment_gap": "Work-structure indicator; raw FT/PT counts used to compute F-M incidence gap",
            "women_senior_middle_management_share": "Leadership proxy indicator",
        }[key],
        "notes": {
            "stem_graduates_women_share": "Direct OBS_VALUE used as women among tertiary STEM graduates.",
            "coding_gap_youth": "Uses ages 16-24 and percentage of population; final gap is male minus female.",
            "labour_force_participation_gap": "Uses ages 15-74 to align with OECD Gender Dashboard; final gap is male minus female.",
            "gender_wage_gap": "Median gender wage gap; direct OBS_VALUE.",
            "part_time_employment_gap": "Uses persons counts for FT/PT employees; final gap is female minus male part-time incidence in percentage points.",
            "women_senior_middle_management_share": "Proxy for leadership power; direct OBS_VALUE from SDG series for senior and middle management.",
        }[key],
    }
    for key, url in api_urls.items()
])
source_url_registry.to_csv(raw_dir / "source_url_registry.csv", index=False)

saved_files = pd.DataFrame({
    "file": [
        "data/processed/gender_conversion_gap_latest.csv",
        "data/processed/indicator_coverage.csv",
        "data/processed/pairwise_overlap.csv",
        "data/processed/data_dictionary.csv",
        "data/processed/hypothesis_mapping.csv",
        "data/processed/estonia_latest_values.csv",
        "data/raw/source_url_registry.csv",
        "data/processed/raw_missingness_summary.csv",
    ]
})

print("Saved CSV files:")
display(saved_files)



Saved CSV files:


,file
0,data/processed/gender_conversion_gap_latest.csv
1,data/processed/indicator_coverage.csv
2,data/processed/pairwise_overlap.csv
3,data/processed/data_dictionary.csv
4,data/processed/hypothesis_mapping.csv
5,data/processed/estonia_latest_values.csv
6,data/raw/source_url_registry.csv
7,data/processed/raw_missingness_summary.csv
